# 61 ? A retrieval: longer training and measured comparison

Your reported baseline is **Micro-F1@7 = 0.192329**, with 817 training queries and 118 dev queries.
You identified too little training as the issue. This notebook reads the actual saved configuration/logs,
then prioritizes a **30-epoch candidate** with renewed training negatives and dev-best checkpoint selection.
With the earlier default 32-query batch and 3 epochs, the schedule has approximately 78 optimizer updates;
30 epochs offers approximately 780, subject to early stopping. Actual counts are shown below.

The first recipe preserves the source representation, positive/negative counts, temperature, and chosen
effective batch. It uses LR 3e-5, 3% warmup and six-evaluation patience. These settings are proposed
candidates, not a measured optimum. The budget, schedule and sampling change together; this is not a
single-factor ablation. It starts afresh from the same Phase-1 weights, not from a random model.
It is not an optimizer resume of the earlier PCR checkpoint.

Optional follow-ups add mixed lexical-hard/random TRAIN negatives and larger source coverage.
They change multiple settings, so do not attribute their outcomes to a single factor.
Every comparison uses the same training queries, official dev query subset and dev candidate pool.
The selection step retains the original A checkpoint when candidates tie or regress.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
%pip -q install numpy pandas tokenizers ijson scikit-learn "huggingface_hub[hf_xet]"
# Install before importing model modules. Restart if pip requests it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, glob, zipfile, tempfile, importlib, shutil, time, json
from pathlib import Path
MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_ROOT = MYDRIVE / 'cs781'
RUNS_DIR = DRIVE_ROOT / 'runs'
OUT_DIR = DRIVE_ROOT / 'pcr_runs'
P7 = Path('/content/p7')
P7.mkdir(exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional explicit folder or archive; leave None to find an updated Phase-7 code copy.
CODE_SOURCE = None
levels = ['', '*/', '*/*/', '*/*/*/']
if CODE_SOURCE is None:
    folders = [Path(h).parents[2] for level in levels
               for h in glob.glob(str(MYDRIVE / f'{level}lm_lib/pcr/tuning.py'))]
    archives = [Path(h) for level in levels for h in glob.glob(str(MYDRIVE / f'{level}cs781_code*.zip'))]
    valid_archives = []
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            if 'lm_lib/pcr/tuning.py' in z.namelist():
                valid_archives.append(archive)
    candidates = [(p, (p / 'lm_lib/pcr/tuning.py').stat().st_mtime) for p in folders]
    candidates += [(p, p.stat().st_mtime) for p in valid_archives]
    if not candidates:
        raise FileNotFoundError('Upload updated lm_lib/, models/, adapters/ folders, or rebuild and upload cs781_code.zip.')
    CODE_SOURCE = max(candidates, key=lambda item: item[1])[0]
CODE_SOURCE = Path(CODE_SOURCE)
if CODE_SOURCE.is_file():
    CODE_DIR = Path(tempfile.mkdtemp(prefix='cs781_phase7_', dir='/content'))
    with zipfile.ZipFile(CODE_SOURCE) as z:
        z.extractall(CODE_DIR)
else:
    CODE_DIR = CODE_SOURCE
assert (CODE_DIR / 'lm_lib/pcr/tuning.py').is_file(), 'Selected code copy lacks Phase 7'
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
# Drop project modules cached from a previous phase/archive so imports use this code copy.
for name in list(sys.modules):
    if name in ('lm_lib', 'models', 'adapters') or name.startswith(('lm_lib.', 'models.', 'adapters.')):
        del sys.modules[name]
# Re-run the later import/configuration cells after re-running setup.
print('Code:', CODE_SOURCE, '\nRuns:', OUT_DIR)

In [ ]:
from huggingface_hub import hf_hub_download, login

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    token_file = CODE_DIR / 'hf_token.txt'
    if token_file.is_file():
        return token_file.read_text().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or the environment, or provide hf_token.txt beside the code.') from exc

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
# These paths and the record schema are specified in the bundled brief ?13.3.
cache_dir = DRIVE_ROOT / 'data' / 'phase7'
cache_dir.mkdir(parents=True, exist_ok=True)
for relative in ['Train/train.json', 'Train/train_candidates.json',
                 'Dev/dev.json', 'Dev/dev_candidates.json',
                 'Test/test.json', 'Test/test_candidates.json']:
    name = Path(relative).name
    cached = cache_dir / name
    if not cached.is_file():
        downloaded = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_7_PCR/{relative}',
                                     repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(downloaded, cached)
    shutil.copyfile(cached, P7 / name)
    print(name, f'{(P7 / name).stat().st_size / 1e6:.1f} MB')

In [ ]:
import math
import pandas as pd
from dataclasses import asdict, replace
from lm_lib.pcr.trainer import PCRConfig, train_pcr, predict_test
from lm_lib.pcr.tuning import a_tuning_configs, select_best_a_run
from lm_lib.pcr import data as pd_data

BASELINE_DIR = OUT_DIR / 'A_full__firstlast2x256'
assert (BASELINE_DIR / 'results.json').is_file(), 'Point BASELINE_DIR to your completed A baseline'
base = PCRConfig(**json.loads((BASELINE_DIR / 'config.json').read_text()))
base = replace(base, data_dir=str(P7), out_dir=str(OUT_DIR))
baseline_result = json.loads((BASELINE_DIR / 'results.json').read_text())
baseline_manifest = json.loads((BASELINE_DIR / 'manifest.json').read_text())
assert base.method == 'full' and not base.kd_teacher
print('Baseline actual Micro-F1@7:', baseline_result['best_dev_micro_f1_at7'])
print('Baseline optimizer steps:', baseline_result.get('optimizer_steps'))
print('Baseline epoch budget:', base.epochs, '| LR:', base.lr)
print('Baseline source token budget:', base.chunk_len * base.max_chunks)
if (BASELINE_DIR / 'log.csv').is_file():
    display(pd.read_csv(BASELINE_DIR / 'log.csv'))
# Match the release and dev query IDs to the run that produced the reported baseline.
for split in ['train', 'dev']:
    queries, pool = pd_data.load_split(P7, split)
    assert pd_data.fingerprint(queries, pool) == baseline_manifest[f'{split}_fingerprint'], 'Dataset differs from baseline'
    del queries, pool

In [ ]:
recipes = a_tuning_configs(base)
recipe_rows = []
for name, candidate in recipes.items():
    effective = candidate.effective_batch_queries or candidate.train_batch_size * candidate.grad_accum
    recipe_rows.append(dict(recipe=name, run=candidate.run_name, epochs=candidate.epochs, lr=candidate.lr,
                            max_optimizer_steps=math.ceil(baseline_result['n_train'] / effective) * candidate.epochs,
                            source_tokens=candidate.chunk_len * candidate.max_chunks, strategy=candidate.strategy,
                            token_pooling=candidate.token_pooling, negatives=candidate.negatives,
                            max_positives=candidate.max_positives, negative_sampling=candidate.negative_sampling,
                            effective_queries=effective, patience=candidate.early_stopping_patience))
display(pd.DataFrame(recipe_rows))
# Start with more training. Run optional follow-ups only after reviewing its loss/dev curve.
RUN_RECIPES = ['more_steps']
# RUN_RECIPES = ['more_steps', 'hard_negatives', 'long_context']

`more_steps` uses a 30-epoch budget (or at least four times a longer existing budget), refreshing random
negatives and positive subsamples each epoch. The run stops after six dev evaluations without a strict
F1 improvement; best weights survive. Set `early_stopping_patience=0` on a newly named recipe if you
intentionally want the full budget regardless of plateaus. Inspect the learning curve before doing so.

`hard_negatives` mixes up to six of twelve negatives from a lexical-hard pool and samples the rest
randomly, with up to four positives per query. TF-IDF fits **only TRAIN candidate text** and excludes
all gold positives from hard/random negatives. Pools and per-epoch tuples are saved. Dev/test documents
or relevance labels are never used for mining. Resampling is disabled for cached-score KD because its
candidate tuples must stay aligned.

`long_context` additionally uses four 512-token spread chunks, last-token pooling and an effective batch
of 16. These are bounded case representations, not full-context attention over 2,048 tokens. Auto batch
batch sizing is repeated for each configuration while preserving its effective query batch.

These follow-ups are extra A experiments; they do not replace the phase's required adaptation techniques
and KD. Repeated selection on the same 118 dev queries is optimistic; test labels remain private.

In [ ]:
completed_candidates = []
for name in RUN_RECIPES:
    candidate = recipes[name]
    target = OUT_DIR / candidate.run_name
    if (target / 'results.json').is_file():
        saved = PCRConfig(**json.loads((target / 'config.json').read_text()))
        # Calibration changes physical batch/accumulation, not recipe semantics or effective batch.
        hardware_fields = {'auto_batch', 'batch_calibration', 'train_batch_size', 'grad_accum', 'embed_batch_size'}
        for key, value in asdict(candidate).items():
            if key not in hardware_fields:
                assert asdict(saved)[key] == value, f'Existing {target.name} differs at {key}; use a new run_name'
        print('Reusing completed candidate:', target)
    else:
        assert not target.exists() or not any(target.iterdir()), 'Incomplete run exists; use a fresh run_name'
        print('Training:', name, '| requested epochs:', candidate.epochs)
        train_pcr(candidate)
    completed_candidates.append(str(target))

In [ ]:
# Selection includes the measured original baseline. Ties retain it; no improvement is fabricated.
selection = select_best_a_run(str(BASELINE_DIR), completed_candidates)
(OUT_DIR / 'A_tuning_selection.json').write_text(json.dumps(selection, indent=2))
display(pd.DataFrame(selection['comparisons']).sort_values('micro_f1_at7', ascending=False))
print('Selected run:', selection['selected_run'])
print('Baseline score:', selection['baseline_score'])
print('Selected score:', selection['selected_score'], '| change:', selection['delta'])

In [ ]:
# Inspect the chosen candidate's full learning curve and its best-checkpoint epoch.
for folder in [BASELINE_DIR] + [Path(p) for p in completed_candidates]:
    print(folder.name)
    display(pd.read_csv(folder / 'log.csv'))
# Export test predictions from the measured champion, which may still be the baseline.
zip_path, predictions = predict_test(selection['selected_run'], data_dir=str(P7))
print('Submission:', zip_path)
# Optional:
# from google.colab import files
# files.download(zip_path)

Keep the original A run, candidate configurations, logs, per-epoch training pairs, dev predictions and
`A_tuning_selection.json`. Report measured outcomes yourself; the notebook does not promise a higher
score before running the experiment. The Phase-1 architectures remain unchanged. All recipe runs initialize from
the same original A weights, enabling comparison against your prior A fine-tuning run.